# Overnight Confirmatory Batch

**Goal:** exhaust the outstanding confirmatory experiments in one run, so
tomorrow's decisions can be made from a complete picture rather than a
partial one.

**Design principle: fault tolerance over completeness.** Every experiment
section below is wrapped in try/except and saves its own CSV immediately on
completion. A failure in one section prints the error and moves on to the
next -- it does NOT halt the notebook. This matters specifically because
this is running unattended: a single bad path or edge case should cost one
result, not the whole night.

**Included tonight (priority order):**
1. Chronos determinism check (cheap, foundational -- informs how much to
   trust items 3 and 5)
2. Lorenz phase-surrogate confirmatory rerun (Experiment 14, n=8->n=20) --
   this is this log's single strongest positive finding and has never been
   confirmed at proper sample size
3. Complexity continuum confirmatory rerun -- Duffing + Van der Pol
   (Experiment 19, n=8->n=20). Duffing's original generator was verified
   clean (1.29x growth over the full trajectory, 1.05x within one window --
   compare Harmonic's 42x/1.37x) before this notebook was built, so no
   generator fix is needed here, unlike Harmonic.
4. ETTh2 H=336 replication (the one isolated significant ETT result in this
   log, p=0.013, flagged as exploratory pending replication)
5. G4 -- classical baseline column (seasonal-naive + linear-trend) on
   Weather/ETTh1/ETTh2, compared against Experiment 8's already-logged
   Panda/Chronos numbers (not rerun, to save compute)
6. B3a extended grid + second seed, following up the periods_in_window=1
   lead from the earlier session

**Deliberately excluded tonight, flagged for a separate session:**
- G1 (correlation-dimension revalidation) -- requires a from-scratch
  estimator validated against known ground truth per this log's own
  estimator-validation rule. Building and validating that unattended,
  overnight, with no one to sanity-check intermediate output, is exactly
  the wrong way to run it.
- A1/A2a/A3's Harmonic-specific re-checks -- these need the
  baseline_100k/ablation_100k retrained checkpoints, a different loading
  path than the published-checkpoint pattern used everywhere else tonight.
  Separate setup, separate session.

**Config -- check these two paths before running:**


In [1]:
# ============================================================
# CELL 1 -- CONFIG + IMPORTS + MODEL LOADING
# ============================================================
import os
import json as _json
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

# --- CONFIG: adjust these two if your ts_data layout differs ---
DATA_DIR = './ts_data'
ETTH1_PATH = f'{DATA_DIR}/ETTh1.csv'   # TODO: confirm filename matches your ts_data directory
ETTH2_PATH = f'{DATA_DIR}/ETTh2.csv'   # TODO: confirm filename matches your ts_data directory
WEATHER_PATH = f'{DATA_DIR}/weather.csv'

OUTPUT_DIR = './overnight_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print("Models loaded.")

# --- Run-log: every section appends its status here, printed at the end ---
RUN_LOG = []
def log_status(section, status, detail=""):
    RUN_LOG.append({"section": section, "status": status, "detail": detail})
    print(f"[{status}] {section}: {detail}")


Device: cpu
Models loaded.


In [2]:
# ============================================================
# CELL 2 -- EVALUATION HARNESS (verbatim, unchanged from prior notebooks)
# ============================================================
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def instance_norm_window(x_CT):
    """Degenerate-channel guard added here (B3c/Experiment 41 already found and
    fixed this exact bug -- carried forward now, was missing from this notebook's
    first draft). A near-zero-variance channel (e.g. Weather's 'rain (mm)', exactly
    0 for a full 512-step window during dry periods) with the naive std+1e-8
    divisor blows tiny fluctuations up to enormous normalized values -- this
    produced a ~496.8 mean MAE against a ~0.585 median in Experiment 41 before
    being caught. Channels with std < 1e-6 are treated as unscaled (divide by 1.0)
    rather than divided by a near-zero number."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std_raw = x_CT.std(axis=1, keepdims=True)
    std = np.where(std_raw < 1e-6, 1.0, std_raw)
    return (x_CT - mu) / std, mu, std

CONTEXT_LEN = 512

def panda_forecast(context_np, horizon):
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)

def chronos_forecast(context_np, horizon):
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()

def evaluate(data_CT, horizon, n_windows=20, label="",
             fn_a=None, fn_b=None,
             name_a="panda", name_b="chronos"):
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: T={T} too short")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std

        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        if np.any(diff != 0):
            _, pval = wilcoxon(diff, alternative="greater")
        else:
            pval = 1.0
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    rel_skill = np.median(mae_b) / np.median(mae_a) if np.median(mae_a) > 0 else np.nan

    result = {
        "label"         : label,
        "horizon"       : horizon,
        f"{name_a}_mae" : np.median(mae_a),
        f"{name_a}_iqr" : np.percentile(mae_a,75)-np.percentile(mae_a,25),
        f"{name_b}_mae" : np.median(mae_b),
        f"{name_b}_iqr" : np.percentile(mae_b,75)-np.percentile(mae_b,25),
        "advantage_mae" : adv,
        "rel_skill"     : rel_skill,
        "wilcoxon_p"    : pval,
    }
    sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")
    print(f"  {label:28s} H={horizon:4d}  {name_a}={np.median(mae_a):.4f}  "
          f"{name_b}={np.median(mae_b):.4f}  Adv={adv:+.4f}  p={pval:.4f}{sig}")
    return result

def load_ts(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

print("Harness defined.")


Harness defined.


In [12]:
# ============================================================
# CELL 3 -- SECTION 1/6: CHRONOS DETERMINISM CHECK
# Flagged as an open methodological question in B3b, never directly tested.
# Cheap, foundational: informs how much noise to expect in any Chronos-side
# replication tonight (items 4 and, partially, 2/3).
# ============================================================
try:
    print("=== Section 1: Chronos determinism check ===")
    _rng = np.random.default_rng(0)
    _test_ctx = _rng.standard_normal((3, CONTEXT_LEN)).astype(np.float32)
    _test_ctx_norm, _, _ = instance_norm_window(_test_ctx)

    _out1 = chronos_forecast(_test_ctx_norm, 96)
    _out2 = chronos_forecast(_test_ctx_norm, 96)
    _max_diff = np.abs(_out1 - _out2).max()
    _deterministic = _max_diff == 0.0

    print(f"Max abs difference across two identical calls: {_max_diff}")
    print(f"Deterministic: {_deterministic}")

    pd.DataFrame([{"max_abs_diff": _max_diff, "deterministic": _deterministic}]) \
        .to_csv(f"{OUTPUT_DIR}/01_chronos_determinism.csv", index=False)
    log_status("Chronos determinism check", "OK",
               f"deterministic={_deterministic}, max_diff={_max_diff}")
except Exception as e:
    log_status("Chronos determinism check", "FAILED", str(e))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


=== Section 1: Chronos determinism check ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Max abs difference across two identical calls: 0.4772549271583557
Deterministic: False
[OK] Chronos determinism check: deterministic=False, max_diff=0.4772549271583557


In [3]:
# ============================================================
# CELL 3 -- SECTION 1/6: CHRONOS DETERMINISM CHECK
# Flagged as an open methodological question in B3b, never directly tested.
# Cheap, foundational: informs how much noise to expect in any Chronos-side
# replication tonight (items 4 and, partially, 2/3).
# ============================================================
try:
    print("=== Section 1: Chronos determinism check ===")
    _rng = np.random.default_rng(0)
    _test_ctx = _rng.standard_normal((3, CONTEXT_LEN)).astype(np.float32)
    _test_ctx_norm, _, _ = instance_norm_window(_test_ctx)

    _out1 = chronos_forecast(_test_ctx_norm, 96)
    _out2 = chronos_forecast(_test_ctx_norm, 96)
    _max_diff = np.abs(_out1 - _out2).max()
    _deterministic = _max_diff == 0.0

    print(f"Max abs difference across two identical calls: {_max_diff}")
    print(f"Deterministic: {_deterministic}")

    pd.DataFrame([{"max_abs_diff": _max_diff, "deterministic": _deterministic}]) \
        .to_csv(f"{OUTPUT_DIR}/01_chronos_determinism.csv", index=False)
    log_status("Chronos determinism check", "OK",
               f"deterministic={_deterministic}, max_diff={_max_diff}")
except Exception as e:
    log_status("Chronos determinism check", "FAILED", str(e))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


=== Section 1: Chronos determinism check ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Max abs difference across two identical calls: 0.4065505564212799
Deterministic: False
[OK] Chronos determinism check: deterministic=False, max_diff=0.4065505564212799


In [4]:
# ============================================================
# CELL 4 -- LORENZ GENERATOR + PHASE SURROGATE (verbatim gate_3ch protocol)
# ============================================================
SEED = 42

def simulate_lorenz_gate(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3):
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n - 1):
        k1x = sigma * (y - x); k1y = x * (rho - z) - y; k1z = x * y - beta * z
        k2x = sigma * ((y + dt/2*k1y) - (x + dt/2*k1x))
        k2y = (x + dt/2*k1x) * (rho - (z + dt/2*k1z)) - (y + dt/2*k1y)
        k2z = (x + dt/2*k1x) * (y + dt/2*k1y) - beta * (z + dt/2*k1z)
        k3x = sigma * ((y + dt/2*k2y) - (x + dt/2*k2x))
        k3y = (x + dt/2*k2x) * (rho - (z + dt/2*k2z)) - (y + dt/2*k2y)
        k3z = (x + dt/2*k2x) * (y + dt/2*k2y) - beta * (z + dt/2*k2z)
        k4x = sigma * ((y + dt*k3y) - (x + dt*k3x))
        k4y = (x + dt*k3x) * (rho - (z + dt*k3z)) - (y + dt*k3y)
        k4z = (x + dt*k3x) * (y + dt*k3y) - beta * (z + dt*k3z)
        x += dt/6*(k1x+2*k2x+2*k3x+k4x)
        y += dt/6*(k1y+2*k2y+2*k3y+k4y)
        z += dt/6*(k1z+2*k2z+2*k3z+k4z)
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs]).T

def load_lorenz():
    traj = simulate_lorenz_gate(n=5000)[500:].T  # (3, 4500)
    return traj

def phase_randomize_surrogate(x_1d, seed=None):
    """Verbatim Experiment 14 method: FFT magnitudes preserved, phases
    replaced with uniform random draws, rescaled to match original mean/std
    exactly."""
    rng = np.random.default_rng(seed)
    n = len(x_1d)
    fft_vals = np.fft.rfft(x_1d)
    magnitudes = np.abs(fft_vals)
    random_phases = rng.uniform(0, 2*np.pi, size=len(fft_vals))
    random_phases[0] = 0
    if n % 2 == 0:
        random_phases[-1] = 0
    new_fft = magnitudes * np.exp(1j * random_phases)
    surrogate = np.fft.irfft(new_fft, n=n)
    surrogate = (surrogate - surrogate.mean()) / surrogate.std() * x_1d.std() + x_1d.mean()
    return surrogate.astype(np.float32)

def load_lorenz_surrogate(seed=SEED):
    base = load_lorenz()
    return np.stack([phase_randomize_surrogate(base[c], seed=seed + c)
                      for c in range(base.shape[0])])

print("Lorenz generators defined.")


Lorenz generators defined.


In [5]:
# ============================================================
# CELL 5 -- SECTION 2/6: LORENZ PHASE-SURROGATE CONFIRMATORY RERUN
# Experiment 14 original: n=8, H=96, Panda advantage +0.3835 (chaotic) vs
# +0.1715 (surrogate, p=0.320, not significant at n=8). Decomposition found
# Panda degrades 13.0x vs Chronos's 2.0x on the surrogate -- this project's
# single strongest positive finding. Never confirmed at n=20.
# ============================================================
try:
    print("=== Section 2: Lorenz phase-surrogate confirmatory rerun ===")
    N_WINDOWS = 20
    HORIZONS = [96, 192, 336]

    lorenz_data = load_lorenz()
    lorenz_surrogate_data = load_lorenz_surrogate()

    lorenz_results = []
    for H in HORIZONS:
        r_chaotic = evaluate(lorenz_data, H, n_windows=N_WINDOWS, label=f"lorenz_chaotic_H{H}")
        if r_chaotic:
            r_chaotic["condition"] = "chaotic"
            lorenz_results.append(r_chaotic)
        r_surrogate = evaluate(lorenz_surrogate_data, H, n_windows=N_WINDOWS, label=f"lorenz_surrogate_H{H}")
        if r_surrogate:
            r_surrogate["condition"] = "surrogate"
            lorenz_results.append(r_surrogate)

    df_lorenz = pd.DataFrame(lorenz_results)
    df_lorenz.to_csv(f"{OUTPUT_DIR}/02_lorenz_phase_surrogate.csv", index=False)

    h96 = df_lorenz[df_lorenz["horizon"] == 96]
    chaotic_adv = h96[h96["condition"]=="chaotic"]["advantage_mae"].iloc[0]
    surrogate_adv = h96[h96["condition"]=="surrogate"]["advantage_mae"].iloc[0]
    surrogate_p = h96[h96["condition"]=="surrogate"]["wilcoxon_p"].iloc[0]
    print(f"\nH=96: chaotic advantage={chaotic_adv:+.4f}, surrogate advantage={surrogate_adv:+.4f} (p={surrogate_p:.4f})")
    print(f"Original (n=8): chaotic +0.3835, surrogate +0.1715 (p=0.320)")

    log_status("Lorenz phase-surrogate rerun", "OK",
               f"H=96 chaotic_adv={chaotic_adv:+.4f}, surrogate_adv={surrogate_adv:+.4f}, p={surrogate_p:.4f}")
except Exception as e:
    log_status("Lorenz phase-surrogate rerun", "FAILED", str(e))


=== Section 2: Lorenz phase-surrogate confirmatory rerun ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  lorenz_chaotic_H96           H=  96  panda=0.0498  chronos=0.5335  Adv=+0.4837  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  lorenz_surrogate_H96         H=  96  panda=0.7840  chronos=1.0361  Adv=+0.2520  p=0.0000 *


...

  lorenz_surrogate_H192        H= 192  panda=0.9319  chronos=1.1975  Adv=+0.2656  p=0.0001 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  lorenz_chaotic_H336          H= 336  panda=0.5173  chronos=1.0305  Adv=+0.5133  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  lorenz_surrogate_H336        H= 336  panda=0.9700  chronos=1.2362  Adv=+0.2661  p=0.0000 *

H=96: chaotic advantage=+0.4837, surrogate advantage=+0.2520 (p=0.0000)
Original (n=8): chaotic +0.3835, surrogate +0.1715 (p=0.320)
[OK] Lorenz phase-surrogate rerun: H=96 chaotic_adv=+0.4837, surrogate_adv=+0.2520, p=0.0000


In [6]:
# ============================================================
# CELL 6 -- DUFFING + VAN DER POL GENERATORS
# Both verbatim from eval-nb.ipynb, ORIGINAL protocol (not the skew40-retuned
# versions). Duffing's growth was empirically verified clean BEFORE this
# notebook was built (1.29x over full trajectory, 1.05x within one window --
# see conversation), so it is used as-is, no fix needed. Van der Pol was
# already solve_ivp-based in the original protocol.
# ============================================================
def simulate_duffing(n_steps=4000, delta=0.3, alpha=-1.0, beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    rng = np.random.default_rng(seed)
    dt = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32)

def load_duffing():
    return simulate_duffing()[500:][None, :]

def simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED):
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                     t_eval=np.linspace(0, n_steps*0.05, n_steps),
                     method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)

def load_vanderpol():
    return simulate_vanderpol()[500:][None, :]

print("Duffing/Van der Pol generators defined.")


Duffing/Van der Pol generators defined.


In [7]:
# ============================================================
# CELL 7 -- SECTION 3/6: COMPLEXITY CONTINUUM CONFIRMATORY RERUN
# Experiment 19 original: n=8, H=96. Duffing +0.214 (p=0.055, NOT significant
# -- just above threshold). Van der Pol +0.011 (p=0.027, significant but tiny,
# largely a floor effect per the relative-skill revision). Harmonic (the third
# leg of this table) already retracted -- see Section 18. These two have never
# been checked at n=20.
# ============================================================
try:
    print("=== Section 3: Complexity continuum confirmatory rerun ===")
    N_WINDOWS = 20
    HORIZONS = [96, 192, 336]

    duffing_data = load_duffing()
    vdp_data = load_vanderpol()

    continuum_results = []
    for H in HORIZONS:
        r = evaluate(duffing_data, H, n_windows=N_WINDOWS, label=f"duffing_H{H}")
        if r:
            r["system"] = "duffing"
            continuum_results.append(r)
        r = evaluate(vdp_data, H, n_windows=N_WINDOWS, label=f"van_der_pol_H{H}")
        if r:
            r["system"] = "van_der_pol"
            continuum_results.append(r)

    df_continuum = pd.DataFrame(continuum_results)
    df_continuum.to_csv(f"{OUTPUT_DIR}/03_complexity_continuum.csv", index=False)

    h96 = df_continuum[df_continuum["horizon"] == 96]
    duffing_adv = h96[h96["system"]=="duffing"]["advantage_mae"].iloc[0]
    duffing_p = h96[h96["system"]=="duffing"]["wilcoxon_p"].iloc[0]
    vdp_adv = h96[h96["system"]=="van_der_pol"]["advantage_mae"].iloc[0]
    vdp_p = h96[h96["system"]=="van_der_pol"]["wilcoxon_p"].iloc[0]
    print(f"\nH=96: Duffing advantage={duffing_adv:+.4f} (p={duffing_p:.4f}), "
          f"original was +0.214 (p=0.055, n=8)")
    print(f"H=96: Van der Pol advantage={vdp_adv:+.4f} (p={vdp_p:.4f}), "
          f"original was +0.011 (p=0.027, n=8)")

    log_status("Complexity continuum rerun", "OK",
               f"Duffing adv={duffing_adv:+.4f}/p={duffing_p:.4f}, VdP adv={vdp_adv:+.4f}/p={vdp_p:.4f}")
except Exception as e:
    log_status("Complexity continuum rerun", "FAILED", str(e))


=== Section 3: Complexity continuum confirmatory rerun ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  duffing_H96                  H=  96  panda=0.3879  chronos=0.6601  Adv=+0.2722  p=0.0120 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  van_der_pol_H96              H=  96  panda=0.0350  chronos=0.0405  Adv=+0.0055  p=0.5218


...

  van_der_pol_H192             H= 192  panda=0.0512  chronos=0.0837  Adv=+0.0324  p=0.0181 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  duffing_H336                 H= 336  panda=0.8689  chronos=0.9845  Adv=+0.1156  p=0.0664 ~


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  van_der_pol_H336             H= 336  panda=0.0789  chronos=0.1574  Adv=+0.0786  p=0.0060 *

H=96: Duffing advantage=+0.2722 (p=0.0120), original was +0.214 (p=0.055, n=8)
H=96: Van der Pol advantage=+0.0055 (p=0.5218), original was +0.011 (p=0.027, n=8)
[OK] Complexity continuum rerun: Duffing adv=+0.2722/p=0.0120, VdP adv=+0.0055/p=0.5218


In [8]:
# ============================================================
# CELL 8 -- SECTION 4/6: ETTh2 H=336 REPLICATION
# Experiment 8: n=20, advantage +0.185, p=0.013 -- the one isolated
# significant ETT result, flagged as exploratory (one of eight ETT tests,
# uncorrected) pending replication.
# ============================================================
try:
    print("=== Section 4: ETTh2 H=336 replication ===")
    etth2_data = load_ts(ETTH2_PATH)
    r = evaluate(etth2_data, 336, n_windows=20, label="ETTh2_H336_replication")
    if r:
        pd.DataFrame([r]).to_csv(f"{OUTPUT_DIR}/04_etth2_h336_replication.csv", index=False)
        print(f"\nThis run: advantage={r['advantage_mae']:+.4f}, p={r['wilcoxon_p']:.4f}")
        print(f"Original (Experiment 8, n=20): advantage=+0.185, p=0.013")
        log_status("ETTh2 H=336 replication", "OK",
                   f"adv={r['advantage_mae']:+.4f}, p={r['wilcoxon_p']:.4f}")
    else:
        log_status("ETTh2 H=336 replication", "SKIPPED", "evaluate() returned None")
except Exception as e:
    log_status("ETTh2 H=336 replication", "FAILED", str(e))


=== Section 4: ETTh2 H=336 replication ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh2_H336_replication       H= 336  panda=0.9220  chronos=0.9829  Adv=+0.0609  p=0.0884 ~

This run: advantage=+0.0609, p=0.0884
Original (Experiment 8, n=20): advantage=+0.185, p=0.013
[OK] ETTh2 H=336 replication: adv=+0.0609, p=0.0884


In [9]:
# ============================================================
# CELL 9 -- SECTION 5/6: G4 CLASSICAL BASELINE COLUMN
# No Panda/Chronos calls -- compared against Experiment 8's ALREADY-LOGGED
# numbers, not rerun, to save compute. Two baselines: seasonal-naive (repeat
# the last full period) and linear-trend (per-channel linear fit on context,
# extrapolated).
# ============================================================
try:
    print("=== Section 5: G4 classical baseline column ===")

    def seasonal_naive_forecast(ctx_norm, horizon, period):
        C, T = ctx_norm.shape
        last_period = ctx_norm[:, -period:]
        n_tiles = int(np.ceil(horizon / period))
        tiled = np.tile(last_period, (1, n_tiles))[:, :horizon]
        return tiled

    def linear_trend_forecast(ctx_norm, horizon):
        C, T = ctx_norm.shape
        t = np.arange(T)
        preds = np.zeros((C, horizon))
        for c in range(C):
            coeffs = np.polyfit(t, ctx_norm[c], deg=1)
            future_t = np.arange(T, T + horizon)
            preds[c] = np.polyval(coeffs, future_t)
        return preds

    def evaluate_baseline(data_CT, horizon, n_windows, baseline_fn, baseline_kwargs, label):
        C, T = data_CT.shape
        max_start = T - CONTEXT_LEN - horizon
        if max_start <= 0:
            return None
        starts = np.linspace(0, max_start, n_windows, dtype=int)
        maes = []
        for s in starts:
            ctx_raw = data_CT[:, s:s+CONTEXT_LEN]
            tgt_raw = data_CT[:, s+CONTEXT_LEN:s+CONTEXT_LEN+horizon]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)
            tgt_norm = (tgt_raw - mu) / std
            pred = baseline_fn(ctx_norm, horizon, **baseline_kwargs)
            maes.append(mae(tgt_norm, pred))
        return {"label": label, "horizon": horizon, "baseline_mae": np.median(maes)}

    DATASETS = {
        "Weather": (load_ts(WEATHER_PATH), 144),   # 10-min data, daily period = 144
        "ETTh1":   (load_ts(ETTH1_PATH), 24),        # hourly data, daily period = 24
        "ETTh2":   (load_ts(ETTH2_PATH), 24),
    }

    baseline_results = []
    for ds_name, (data, period) in DATASETS.items():
        for H in [96, 192, 336]:
            r1 = evaluate_baseline(data, H, 20, seasonal_naive_forecast, {"period": period},
                                    f"{ds_name}_H{H}_seasonal_naive")
            if r1:
                r1["dataset"] = ds_name; r1["method"] = "seasonal_naive"
                baseline_results.append(r1)
                print(f"  {ds_name} H={H} seasonal_naive MAE={r1['baseline_mae']:.4f}")
            r2 = evaluate_baseline(data, H, 20, linear_trend_forecast, {},
                                    f"{ds_name}_H{H}_linear_trend")
            if r2:
                r2["dataset"] = ds_name; r2["method"] = "linear_trend"
                baseline_results.append(r2)
                print(f"  {ds_name} H={H} linear_trend MAE={r2['baseline_mae']:.4f}")

    df_baseline = pd.DataFrame(baseline_results)
    df_baseline.to_csv(f"{OUTPUT_DIR}/05_g4_classical_baselines.csv", index=False)

    print("\nCompare against Experiment 8's logged Panda/Chronos MAE (cited, not rerun):")
    print("  Weather H=96:  Panda=0.6378, Chronos=0.8115")
    print("  Weather H=192: Panda=0.7224, Chronos=0.9582")
    print("  Weather H=336: Panda=0.8481, Chronos=1.0843")
    print("  ETTh1 H=96:    Panda=0.7269, Chronos=0.6633")
    print("  ETTh1 H=336:   Panda=0.8571, Chronos=0.9013")
    print("  ETTh2 H=96:    Panda=0.8736, Chronos=0.9494")
    print("  ETTh2 H=336:   Panda=0.9255, Chronos=1.1101")
    print("If any classical baseline MAE above is LOWER than both Panda and Chronos")
    print("at the same dataset/horizon, that reframes the interpretation of this log's")
    print("ETTh results specifically -- worth flagging explicitly if it happens.")

    log_status("G4 classical baselines", "OK", f"{len(baseline_results)} cells computed")
except Exception as e:
    log_status("G4 classical baselines", "FAILED", str(e))


=== Section 5: G4 classical baseline column ===
  Weather H=96 seasonal_naive MAE=0.7188
  Weather H=96 linear_trend MAE=0.9892
  Weather H=192 seasonal_naive MAE=0.7474
  Weather H=192 linear_trend MAE=1.0433
  Weather H=336 seasonal_naive MAE=0.8465
  Weather H=336 linear_trend MAE=1.0949
  ETTh1 H=96 seasonal_naive MAE=0.7142
  ETTh1 H=96 linear_trend MAE=0.8671
  ETTh1 H=192 seasonal_naive MAE=0.7904
  ETTh1 H=192 linear_trend MAE=1.0187
  ETTh1 H=336 seasonal_naive MAE=0.8721
  ETTh1 H=336 linear_trend MAE=0.9654
  ETTh2 H=96 seasonal_naive MAE=0.7482
  ETTh2 H=96 linear_trend MAE=0.9474
  ETTh2 H=192 seasonal_naive MAE=0.9843
  ETTh2 H=192 linear_trend MAE=1.0045
  ETTh2 H=336 seasonal_naive MAE=0.9727
  ETTh2 H=336 linear_trend MAE=1.2357

Compare against Experiment 8's logged Panda/Chronos MAE (cited, not rerun):
  Weather H=96:  Panda=0.6378, Chronos=0.8115
  Weather H=192: Panda=0.7224, Chronos=0.9582
  Weather H=336: Panda=0.8481, Chronos=1.0843
  ETTh1 H=96:    Panda=0.7269

In [10]:
# ============================================================
# CELL 10 -- SECTION 6/6: B3a EXTENDED GRID + SECOND SEED
# Follows up the periods_in_window=1 lead from the earlier session. Denser
# grid around the two extremes where the actual signal appeared to live,
# plus a second seed to rule out single-trajectory idiosyncrasy (the same
# concern already documented for Lorenz). Uses the STABLE (solve_ivp)
# Harmonic generator, consistent with the B3a results being extended --
# NOT the original Euler generator (which is known unreliable, see Section 18).
# ============================================================
try:
    print("=== Section 6: B3a extended grid + second seed ===")

    def simulate_harmonic_stable(omega=1.0, dt=0.05, n_steps=4000, seed=SEED):
        rng = np.random.default_rng(seed)
        def rhs(t, y):
            x, v = y
            return [v, -omega**2 * x]
        ic = [float(rng.standard_normal()), float(rng.standard_normal())]
        sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                         t_eval=np.linspace(0, n_steps*dt, n_steps),
                         method='RK45', rtol=1e-8, atol=1e-8)
        return sol.y[0].astype(np.float32)

    def omega_for_periods_in_window(piw, dt=0.05, context_len=CONTEXT_LEN):
        return piw * 2 * np.pi / (context_len * dt)

    def load_harmonic_stable(omega=1.0, seed=SEED):
        series = simulate_harmonic_stable(omega=omega, seed=seed)
        return series[500:][None, :]

    PIW_GRID_EXTENDED = [0.5, 0.75, 1, 1.5, 2, 4, 8, 12, 16, 24, 32, 48]
    SEEDS = [42, 123]

    b3a_results = []
    for seed in SEEDS:
        for piw in PIW_GRID_EXTENDED:
            omega = omega_for_periods_in_window(piw)
            data = load_harmonic_stable(omega=omega, seed=seed)
            r = evaluate(data, 96, n_windows=20, label=f"piw={piw}_seed={seed}")
            if r:
                r["periods_in_window"] = piw
                r["seed"] = seed
                b3a_results.append(r)

    df_b3a_ext = pd.DataFrame(b3a_results)
    df_b3a_ext.to_csv(f"{OUTPUT_DIR}/06_b3a_extended.csv", index=False)

    # Quick cross-seed stability check at the two extremes
    for piw_check in [0.5, 1, 48]:
        rows = df_b3a_ext[df_b3a_ext["periods_in_window"] == piw_check]
        if len(rows) == 2:
            skills = rows["rel_skill"].values
            print(f"  piw={piw_check}: rel_skill seed42={skills[0]:.2f}, "
                  f"seed123={skills[1]:.2f}, ratio={max(skills)/min(skills):.2f}x")

    log_status("B3a extended grid", "OK", f"{len(b3a_results)} cells across {len(SEEDS)} seeds")
except Exception as e:
    log_status("B3a extended grid", "FAILED", str(e))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


=== Section 6: B3a extended grid + second seed ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  piw=0.5_seed=42              H=  96  panda=0.0172  chronos=0.0879  Adv=+0.0708  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

  piw=24_seed=123              H=  96  panda=0.0348  chronos=0.2148  Adv=+0.1800  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  piw=32_seed=123              H=  96  panda=0.1949  chronos=0.0097  Adv=-0.1852  p=1.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  piw=48_seed=123              H=  96  panda=0.0669  chronos=0.0170  Adv=-0.0498  p=0.9999
  piw=0.5: rel_skill seed42=5.12, seed123=5.87, ratio=1.15x
  piw=1: rel_skill seed42=11.67, seed123=15.10, ratio=1.29x
  piw=48: rel_skill seed42=0.24, seed123=0.25, ratio=1.06x
[OK] B3a extended grid: 24 cells across 2 seeds


In [11]:
# ============================================================
# CELL 11 -- FINAL SUMMARY
# Reads back whatever CSVs exist (robust to partial failures) and prints a
# consolidated status report.
# ============================================================
print("="*70)
print("OVERNIGHT BATCH: FINAL STATUS")
print("="*70)
for entry in RUN_LOG:
    print(f"  [{entry['status']:8s}] {entry['section']}: {entry['detail']}")

print()
print("="*70)
print("Files written to", OUTPUT_DIR, ":")
print("="*70)
if os.path.exists(OUTPUT_DIR):
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"  {f}")

print()
print("Wake-up checklist:")
print("  1. Check RUN_LOG above for any FAILED sections -- these need attention first.")
print("  2. Lorenz phase-surrogate (02): does the surrogate advantage stay non-significant")
print("     at n=20, or does it now reach significance? Either way this is informative.")
print("  3. Complexity continuum (03): did Duffing/Van der Pol advantages survive n=20,")
print("     same question the heterogeneity bottleneck and Harmonic both failed.")
print("  4. ETTh2 (04): did the p=0.013 result replicate independently?")
print("  5. G4 (05): does any classical baseline beat BOTH Panda and Chronos anywhere?")
print("  6. B3a (06): does the piw=1 Chronos weak spot hold up across both seeds and")
print("     the denser grid, or was it single-trajectory noise?")


OVERNIGHT BATCH: FINAL STATUS
  [OK      ] Chronos determinism check: deterministic=False, max_diff=0.4065505564212799
  [OK      ] Lorenz phase-surrogate rerun: H=96 chaotic_adv=+0.4837, surrogate_adv=+0.2520, p=0.0000
  [OK      ] Complexity continuum rerun: Duffing adv=+0.2722/p=0.0120, VdP adv=+0.0055/p=0.5218
  [OK      ] ETTh2 H=336 replication: adv=+0.0609, p=0.0884
  [OK      ] G4 classical baselines: 18 cells computed
  [OK      ] B3a extended grid: 24 cells across 2 seeds

Files written to ./overnight_results :
  01_chronos_determinism.csv
  02_lorenz_phase_surrogate.csv
  03_complexity_continuum.csv
  04_etth2_h336_replication.csv
  05_g4_classical_baselines.csv
  06_b3a_extended.csv

Wake-up checklist:
  1. Check RUN_LOG above for any FAILED sections -- these need attention first.
  2. Lorenz phase-surrogate (02): does the surrogate advantage stay non-significant
     at n=20, or does it now reach significance? Either way this is informative.
  3. Complexity continuum (03):